# components/selection_panel

> Selection panel component showing selected files with drag-drop reordering

In [ ]:
#| default_exp components.selection_panel

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Any, Dict, List, Optional
import json

from fasthtml.common import Div, Span, Button, P

from cjm_fasthtml_daisyui.components.actions.button import btn, btn_sizes, btn_styles
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors, badge_sizes, badge_styles
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui

from cjm_fasthtml_tailwind.utilities.spacing import m
from cjm_fasthtml_tailwind.utilities.sizing import w
from cjm_fasthtml_tailwind.utilities.typography import font_size, text_align, truncate
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, justify, items, grow, gap
)
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_lucide_icons.factory import lucide_icon

from cjm_fasthtml_sortable_queue.config import SortableQueueConfig
from cjm_fasthtml_sortable_queue.html_ids import SortableQueueHtmlIds
from cjm_fasthtml_sortable_queue.models import SortableQueueUrls
from cjm_fasthtml_sortable_queue.rendering import render_sortable_queue

from cjm_transcription_source_select.models import SourceSelectUrls, SelectedFile, ExtractionResult
from cjm_transcription_source_select.html_ids import SourceSelectHtmlIds
from cjm_transcription_source_select.utils import format_file_size

## Type Badge

In [ ]:
#| export
def _render_type_badge(
    file_type: str,  # "audio" or "video"
) -> Span:  # Badge component
    """Render a type badge for a selected file."""
    if file_type == "video":
        return Span("video", cls=combine_classes(badge, badge_styles.ghost, badge_sizes.sm, text_dui.secondary))
    return Span("audio", cls=combine_classes(badge, badge_styles.ghost, badge_sizes.sm, text_dui.info))

def _render_extraction_status(
    extraction_result: Optional[ExtractionResult],  # Extraction result for this video file
    file_path: str,  # File path for HTML ID
) -> Optional[Span]:  # Status indicator or None
    """Render extraction status indicator for a video file."""
    if extraction_result is None:
        return Span(
            "Needs extraction",
            id=SourceSelectHtmlIds.extraction_status(file_path),
            cls=combine_classes(font_size.xs, text_dui.warning, m.l(1))
        )

    status = extraction_result.get("status", "pending")
    if status == "complete":
        return Span(
            "Extracted",
            id=SourceSelectHtmlIds.extraction_status(file_path),
            cls=combine_classes(font_size.xs, text_dui.success, m.l(1))
        )
    elif status == "extracting":
        return Span(
            "Extracting...",
            id=SourceSelectHtmlIds.extraction_status(file_path),
            cls=combine_classes(font_size.xs, text_dui.info, m.l(1))
        )
    elif status == "error":
        error_msg = extraction_result.get("error", "Unknown error")
        return Span(
            f"Error: {error_msg}",
            id=SourceSelectHtmlIds.extraction_status(file_path),
            cls=combine_classes(font_size.xs, text_dui.error, m.l(1)),
            title=error_msg
        )
    else:  # pending
        return Span(
            "Needs extraction",
            id=SourceSelectHtmlIds.extraction_status(file_path),
            cls=combine_classes(font_size.xs, text_dui.warning, m.l(1))
        )

## Queue Configuration

In [ ]:
#| export
TSS_QUEUE_PREFIX = "tss"

TSS_QUEUE_CONFIG = SortableQueueConfig(
    prefix=TSS_QUEUE_PREFIX,
    key_field="path",
)

TSS_QUEUE_IDS = SortableQueueHtmlIds(prefix=TSS_QUEUE_PREFIX)

## Queue Content Callback

In [ ]:
#| export
def _make_render_content(
    urls: SourceSelectUrls,  # URL bundle for preview button
    extraction_results: Optional[Dict[str, ExtractionResult]] = None,  # video_path → result
):  # Returns a render_content callback
    """Create a render_content callback with access to URLs and extraction results."""
    def render_content(item: dict, index: int) -> Any:
        """Render filename, type badge, extraction status, size, and preview button."""
        path = item.get("path", "")
        filename = item.get("filename", path.rsplit("/", 1)[-1])
        file_type = item.get("file_type", "audio")
        size_bytes = item.get("size_bytes", 0)

        # Extraction status for video files
        extraction_status = None
        if file_type == "video":
            result = (extraction_results or {}).get(path)
            extraction_status = _render_extraction_status(result, path)

        return Div(
            # Filename
            Span(filename, cls=combine_classes(grow(), font_size.sm, truncate), title=path),
            # Type badge
            _render_type_badge(file_type),
            # Extraction status (video files only)
            extraction_status,
            # Size
            Span(
                format_file_size(size_bytes) if size_bytes else "",
                cls=combine_classes(font_size.xs, text_dui.base_content.opacity(60), m.l(2))
            ),
            # Preview button
            Button(
                lucide_icon("eye", size=4, cls=str(text_dui.base_content.opacity(60))),
                cls=combine_classes(btn, btn_styles.ghost, btn_sizes.xs, m.l(1)),
                hx_post=urls.preview,
                hx_vals=json.dumps({"path": path}),
                hx_target=SourceSelectHtmlIds.as_selector(SourceSelectHtmlIds.PREVIEW_PANEL),
                hx_swap="outerHTML",
                title="Preview file"
            ),
            cls=combine_classes(flex_display, items.center, grow(), gap(1))
        )
    return render_content

## Selection Panel Rendering

In [ ]:
#| export
def _render_queue_empty() -> Any:  # Empty state element
    """Render the custom empty state for the file selection queue."""
    return P(
        "Click files in the browser to select them",
        cls=combine_classes(text_dui.base_content.opacity(30), text_align.center, font_size.xs)
    )


def render_selection_panel(
    selected_files: List[SelectedFile],  # Ordered list of selected files
    urls: SourceSelectUrls,  # URL bundle
    extraction_results: Optional[Dict[str, ExtractionResult]] = None,  # video_path → result
) -> Div:  # Selection panel component
    """Render the selection panel via cjm-fasthtml-sortable-queue."""
    sq_urls = SortableQueueUrls(reorder=urls.reorder, remove=urls.remove, clear=urls.clear)
    return render_sortable_queue(
        config=TSS_QUEUE_CONFIG,
        ids=TSS_QUEUE_IDS,
        urls=sq_urls,
        queue_items=selected_files,
        render_content=_make_render_content(urls, extraction_results),
        render_empty=_render_queue_empty,
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()